In [5]:
import numpy as np
import pandas as pd
import folium
from folium import plugins
import json
from pathlib import Path

# Cấu hình đường dẫn (Thay đổi cho đúng với thư mục của bạn)
BASE_DIR = Path.cwd()
PROCESSED_DIR = Path(
    BASE_DIR
    / ".."
    / ".."
    / ".."
    / "data"
    / "processed"
) 

def load_latest_npz(dataset_name):
    dataset_path = PROCESSED_DIR / dataset_name
    files = sorted(dataset_path.glob(f"{dataset_name}_*.npz"))
    if not files:
        print(f"❌ Không tìm thấy file trong {dataset_path}")
        return None
    print(f"📖 Đang đọc: {files[-1].name}")
    return np.load(files[-1], allow_pickle=True)

In [9]:
# 1. Load OSM Graph
osm_data = load_latest_npz("osm_graph")

if osm_data:
    coords = osm_data['coordinates']  # [N, 2] -> lat, lon
    edge_index = osm_data['edge_index'] # [2, E]
    
    # Khởi tạo bản đồ tại trung tâm Quận 1
    m = folium.Map(location=[coords[:,0].mean(), coords[:,1].mean()], zoom_start=14, tiles="cartodbpositron")
    
    # Vẽ các cạnh của đồ thị
    for i in range(edge_index.shape[1]):
        u_idx, v_idx = edge_index[0, i], edge_index[1, i]
        line = [coords[u_idx].tolist(), coords[v_idx].tolist()]
        folium.PolyLine(line, color="blue", weight=1, opacity=0.5).add_to(m)
    
    print("✅ Đã tạo bản đồ khung OSM.")
    m.save("osm_skeleton.html")

📖 Đang đọc: osm_graph_20260327_135246.npz
✅ Đã tạo bản đồ khung OSM.


In [10]:
# 1. Load Graph Structure đã map-match
graph_struct = load_latest_npz("graph_structure") # Hoặc graph_structure_temporal

if graph_struct:
    coords = graph_struct['coordinates']
    edge_index = graph_struct['edge_index']
    
    # Lấy node_features (ví dụ lấy slot đầu tiên nếu là temporal [N, T, F])
    # Feature index 0 thường là average_speed theo NODE_FEATURE_COLS trong map_matcher.py
    node_feats = graph_struct['node_features'] 
    if node_feats.ndim == 3: # Nếu là [N, T, F]
        speeds = node_feats[:, 0, 0] # Lấy node đầu, slot 0, feature 0 (speed)
    else:
        speeds = node_feats[:, 0]

    m_traffic = folium.Map(location=[coords[:,0].mean(), coords[:,1].mean()], zoom_start=15)

    def get_color(speed):
        if speed > 40: return "green"
        if speed > 20: return "orange"
        return "red"

    # Vẽ từng đoạn đường với màu sắc theo tốc độ
    for i in range(edge_index.shape[1]):
        u_idx, v_idx = edge_index[0, i], edge_index[1, i]
        # Tốc độ trung bình của đoạn đường (trung bình 2 đầu node)
        avg_s = (speeds[u_idx] + speeds[v_idx]) / 2
        
        line = [coords[u_idx].tolist(), coords[v_idx].tolist()]
        folium.PolyLine(
            line, 
            color=get_color(avg_s), 
            weight=4, 
            opacity=0.8,
            tooltip=f"Speed: {avg_s:.2f} km/h"
        ).add_to(m_traffic)

    print("✅ Đã tạo bản đồ lưu lượng giao thông (Green: >40, Orange: 20-40, Red: <20)")
    m_traffic.save("graph_structure.html")

📖 Đang đọc: graph_structure_20260327_152434.npz


KeyError: 'node_features is not a file in the archive'

In [11]:
# Load dữ liệu đã export cho training
model_data = load_latest_npz("model_ready_data")

if model_data:
    X_train = model_data['X_train'] # [Samples, Seq_len, Nodes, Features]
    metadata = json.loads(str(model_data['_metadata'][0]))
    
    print("📊 THÔNG SỐ DATASET:")
    print(f" - Shape X_train: {X_train.shape}")
    print(f" - Số lượng Nodes: {metadata['num_nodes']}")
    print(f" - Số lượng Features: {metadata['num_features']}")
    print(f" - Window Size: {metadata['sequence_length']} ({(metadata['sequence_length']*15)/60} giờ)")
    
    # Hiển thị thử giá trị của 1 node trong 1 window
    first_node_window = X_train[0, :, 0, 0] # Sample 0, All Time, Node 0, Feature 0 (Speed)
    print(f"\n📈 Speed của Node 0 trong window đầu tiên (đã normalize):")
    print(first_node_window)

📖 Đang đọc: model_ready_data_20260327_144235_20260327_144235.npz
📊 THÔNG SỐ DATASET:
 - Shape X_train: (504, 12, 385, 27)
 - Số lượng Nodes: 385
 - Số lượng Features: 27
 - Window Size: 12 (3.0 giờ)

📈 Speed của Node 0 trong window đầu tiên (đã normalize):
[-0.613034   -0.5611586  -0.44443855 -0.7297538  -0.820536   -0.44443855
 -0.5611586  -0.3536566  -0.3147498  -0.6778784  -0.8335047  -0.49631423]
